# M2.2.b: Value-change features (120 features)

Generates 60 shifts x 2 ops = 120 value-change features per plan.
Input: train_features.csv
Output: data/interim/value_change.csv (building_id + timestamp + 120 lag cols)

Formula direction follows plan (not paper Eq.2/3): tree-based invariant.

In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv("../data/raw/train_features.csv")
train = train.sort_values(["building_id", "timestamp"]).reset_index(drop=True)
print(f"Loaded and sorted: {train.shape}")

In [ ]:
shifts = (
    list(np.arange(-24, 0))
    + list(np.arange(1, 25))
    + list(np.arange(-168, -24, 24))
    + list(np.arange(48, 169, 24))
)
print(f"Total shifts: {len(shifts)}")
print(f"Shifts: {sorted(shifts)}")
assert len(shifts) == 60

In [ ]:
for n in shifts:
    train[f"lag_value_{n}"] = (
        train.groupby("building_id")["meter_reading"].shift(n) - train["meter_reading"]
    )
    train[f"lag_value_ratio_{n}"] = (
        train.groupby("building_id")["meter_reading"].shift(n) + 1
    ) / (train["meter_reading"] + 1)

lag_cols = [c for c in train.columns if c.startswith("lag_value_")]
print(f"New lag features: {len(lag_cols)}")
assert len(lag_cols) == 120, f"Expected 120, got {len(lag_cols)}"
print(f"Train shape now: {train.shape}")

In [ ]:
first_bid = train["building_id"].iloc[0]
sample = train[train["building_id"] == first_bid][
    ["timestamp", "meter_reading", "lag_value_1", "lag_value_ratio_1"]
].head(5)
print(f"First building (id={first_bid}) first 5 rows:")
print(sample.to_string())

# Verification: row 1 lag_value_1 = shift(1) - original = row0 - row1
first_bld = train[train["building_id"] == first_bid].reset_index(drop=True)
mr_row0 = first_bld.iloc[0]["meter_reading"]
mr_row1 = first_bld.iloc[1]["meter_reading"]
expected_diff = mr_row0 - mr_row1
actual_diff = first_bld.iloc[1]["lag_value_1"]

print()
print("Verification (row 1):")
print(f"  row 0 meter_reading: {mr_row0}")
print(f"  row 1 meter_reading: {mr_row1}")
print(f"  Expected (row0 - row1): {expected_diff}")
print(f"  Actual lag_value_1: {actual_diff}")
print(f"  Direction correct: {np.isclose(expected_diff, actual_diff, equal_nan=True)}")

In [ ]:
boundary_nans = {
    "lag_value_1": train["lag_value_1"].isna().sum(),
    "lag_value_-1": train["lag_value_-1"].isna().sum(),
    "lag_value_168": train["lag_value_168"].isna().sum(),
    "lag_value_-168": train["lag_value_-168"].isna().sum(),
}
print("NaN counts at boundary shifts:")
for col, count in boundary_nans.items():
    print(f"  {col}: {count:,}")

print()
print("Expected (with 104 incomplete buildings):")
print("  shift +/-1:   ~200 each (1 boundary x 200 buildings)")
print("  shift +/-168: 30,000-45,000 (168 boundaries + gap NaNs across 200 buildings)")

In [ ]:
# Output: value-change features are NOT saved to CSV
# Reason: 1,749,494 rows x 122 cols = ~2.9 GB, too large for interim storage.
# M2.2.e will re-generate by importing the logic from Cells 1-3 of this notebook.
# Cell 3 elapsed 3.3s, so regeneration cost is negligible.
print("Feature generation complete: 120 lag features in memory")
print("Skipping CSV save (file would be ~2.9 GB; regenerate in M2.2.e)")
print("To regenerate, re-run Cells 1-3 of this notebook")